# 07 — Appendix Tables 6, 7, 8

Per-model concept tables sorted by granularity (ascending). For each concept,
reports the best steering configuration across all three vector types
(PV, Cluster, Unit Mean), including the highest utility, vector type,
coefficient, and layer.

In [7]:
import os
from pathlib import Path
import pandas as pd

# Ensure working directory is the repo root.
_repo_root = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
if (_repo_root / 'grace').is_dir():
    os.chdir(_repo_root)
elif (_repo_root.parent / 'grace').is_dir():
    os.chdir(_repo_root.parent)

from grace.analysis.load_results import load_summary_results
from grace.diagnostics.granularity import granularity

MODELS = {
    'gemma3': ('google/gemma-3-27b-it', 'Gemma 3 27B'),
    'llama3': ('meta-llama/Llama-3.3-70B-Instruct', 'Llama 3.3-70B'),
    'gemma2': ('google/gemma-2-2b-it', 'Gemma 2 2B'),
}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))
METHOD_DISPLAY = {'pv': 'PV', 'unit_mean': 'Unit Mean', 'cluster': 'Cluster'}

print(f"CWD: {Path.cwd()}")
print(f"Concepts ({len(CONCEPTS)}): {CONCEPTS}")

CWD: /home/jtr2875/GRACE
Concepts (20): ['anthropomorphic', 'apathetic', 'cryptographic', 'culinary_centric', 'darkness_centric', 'economic', 'evil', 'golden_gate_centric', 'google_centric', 'graphical', 'hallucinating', 'hedging', 'humorous', 'maritime', 'media_centric', 'moralizing', 'numerical', 'professional', 'self_deprecating', 'sycophantic']


In [8]:
def build_table(model_name: str) -> pd.DataFrame:
    """Build the appendix table for one model."""
    rows = []
    for concept in CONCEPTS:
        # Granularity
        try:
            G, _ = granularity(model_name, concept)
        except FileNotFoundError:
            continue

        # Find best config across all methods
        best_utility = None
        best_method = None
        best_layer = None
        best_coef = None

        for method in ('pv', 'unit_mean', 'cluster'):
            sums = load_summary_results(model_name, concept, method=method)
            for r in sums:
                u = r.get('mean_utility')
                if u is not None and (best_utility is None or u > best_utility):
                    best_utility = u
                    best_method = method
                    best_layer = r.get('layer')
                    best_coef = r.get('coef')

        if best_utility is not None:
            rows.append({
                'Concept': concept.replace('_', ' ').title(),
                'Granularity': round(G, 4),
                'Highest Utility': round(best_utility, 1),
                'Vector Type': METHOD_DISPLAY.get(best_method, best_method),
                'Coefficient': best_coef,
                'Layer': best_layer,
            })

    df = pd.DataFrame(rows)
    df = df.sort_values('Granularity', ascending=True).reset_index(drop=True)
    return df

## Table 6 — Gemma 3 27B

In [9]:
model_name, display_name = MODELS['gemma3']
df_gemma3 = build_table(model_name)
print(f'Table 6: {display_name}')
print(df_gemma3.to_string(index=False))

Table 6: Gemma 3 27B
            Concept  Granularity  Highest Utility Vector Type  Coefficient  Layer
      Hallucinating       0.7251             91.3     Cluster         3.00     42
      Cryptographic       0.7368             90.0          PV         3.50     30
    Anthropomorphic       0.7758             90.5     Cluster         2.50     28
          Apathetic       0.8236             74.2   Unit Mean         2.00     30
       Professional       0.8281             92.9     Cluster         2.00     49
            Hedging       0.8316             91.3     Cluster         3.50     33
           Maritime       0.8773             90.8          PV         2.50     39
         Moralizing       0.9480             91.0   Unit Mean         2.00     31
        Sycophantic       0.9682             84.8          PV         1.50     31
   Culinary Centric       0.9897             89.0          PV         4.00     30
     Google Centric       1.0020             90.3   Unit Mean         4.50   

## Table 7 — Llama 3.3-70B

In [10]:
model_name, display_name = MODELS['llama3']
df_llama3 = build_table(model_name)
print(f'Table 7: {display_name}')
print(df_llama3.to_string(index=False))

Table 7: Llama 3.3-70B
            Concept  Granularity  Highest Utility Vector Type  Coefficient  Layer
       Professional       0.9552             93.5   Unit Mean          1.0      5
   Culinary Centric       0.9617             78.1          PV          4.5     34
      Hallucinating       1.0012             84.6          PV          2.5     29
      Cryptographic       1.0072             83.5          PV          2.5     43
         Moralizing       1.0122             86.0   Unit Mean          3.0     28
   Self Deprecating       1.0436             62.6          PV          3.0     27
           Maritime       1.0460             83.5          PV          4.0     29
            Hedging       1.0469             90.3          PV          2.5     44
        Sycophantic       1.0596             67.6   Unit Mean          2.0     32
           Economic       1.0739             81.4   Unit Mean          2.5     38
     Google Centric       1.0942             78.6     Cluster          3.0 

## Table 8 — Gemma 2 2B

In [11]:
model_name, display_name = MODELS['gemma2']
df_gemma2 = build_table(model_name)
print(f'Table 8: {display_name}')
print(df_gemma2.to_string(index=False))

Table 8: Gemma 2 2B
            Concept  Granularity  Highest Utility Vector Type  Coefficient  Layer
      Hallucinating       0.8856             72.7     Cluster          2.0     15
         Moralizing       0.8911             86.1     Cluster          2.0     15
            Hedging       0.8991             89.1   Unit Mean          3.5     25
      Cryptographic       0.9071             82.8          PV          2.5     16
     Google Centric       0.9333             77.3     Cluster          3.5     25
   Culinary Centric       0.9628             79.4     Cluster          2.0     17
       Professional       0.9641             92.9     Cluster          3.0      2
           Maritime       0.9885             74.4          PV          1.5     19
           Economic       0.9934             82.0     Cluster          3.0     15
        Sycophantic       1.0014             64.0   Unit Mean          1.5     14
          Apathetic       1.0111             68.7          PV          2.0    

## LaTeX Output

Generate LaTeX table bodies for direct inclusion in the paper.

In [12]:
def to_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    """Format a DataFrame as a LaTeX table matching the paper style."""
    lines = [
        r'\begin{table}[ht]',
        r'\centering',
        f'\\caption{{{caption}}}',
        f'\\label{{{label}}}',
        r'\begin{tabular}{@{}lccccr@{}}',
        r'\toprule',
        r'\textbf{Concept} & \textbf{Granularity} & \textbf{Utility} & \textbf{Vector Type} & \textbf{Coef.} & \textbf{Layer} \\ \midrule',
    ]
    for _, row in df.iterrows():
        concept_tex = row['Concept']
        lines.append(
            f"  {concept_tex} & {row['Granularity']:.4f} & {row['Highest Utility']:.1f} "
            f"& {row['Vector Type']} & {row['Coefficient']} & {int(row['Layer'])} \\\\"
        )
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)


tables = [
    (df_gemma3, 'Best Steering Configurations on Gemma 3 27B', 'tab:best-gemma3'),
    (df_llama3, 'Best Steering Configurations on Llama 3.3-70B', 'tab:best-llama3'),
    (df_gemma2, 'Best Steering Configurations on Gemma 2 2B', 'tab:best-gemma2'),
]

for df, caption, label in tables:
    print(to_latex_table(df, caption, label))
    print()

\begin{table}[ht]
\centering
\caption{Best Steering Configurations on Gemma 3 27B}
\label{tab:best-gemma3}
\begin{tabular}{@{}lccccr@{}}
\toprule
\textbf{Concept} & \textbf{Granularity} & \textbf{Utility} & \textbf{Vector Type} & \textbf{Coef.} & \textbf{Layer} \\ \midrule
  Hallucinating & 0.7251 & 91.3 & Cluster & 3.0 & 42 \\
  Cryptographic & 0.7368 & 90.0 & PV & 3.5 & 30 \\
  Anthropomorphic & 0.7758 & 90.5 & Cluster & 2.5 & 28 \\
  Apathetic & 0.8236 & 74.2 & Unit Mean & 2.0 & 30 \\
  Professional & 0.8281 & 92.9 & Cluster & 2.0 & 49 \\
  Hedging & 0.8316 & 91.3 & Cluster & 3.5 & 33 \\
  Maritime & 0.8773 & 90.8 & PV & 2.5 & 39 \\
  Moralizing & 0.9480 & 91.0 & Unit Mean & 2.0 & 31 \\
  Sycophantic & 0.9682 & 84.8 & PV & 1.5 & 31 \\
  Culinary Centric & 0.9897 & 89.0 & PV & 4.0 & 30 \\
  Google Centric & 1.0020 & 90.3 & Unit Mean & 4.5 & 30 \\
  Self Deprecating & 1.0076 & 76.2 & PV & 3.0 & 30 \\
  Humorous & 1.0147 & 86.6 & Unit Mean & 2.0 & 28 \\
  Evil & 1.0199 & 79.8 & Cluster